In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import seaborn as sns
sns.set()
import torchvision
import torchvision.transforms as transforms
import umap

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import rand_score
from sklearn.metrics import normalized_mutual_info_score
from mpl_toolkits.mplot3d import Axes3D 

/home/adhin/VSCode/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
file = "./data/weather/tokens_train.pt"
data = torch.load(file, weights_only=False, map_location="cpu", mmap=True)
print(data.keys())
print(data)

dict_keys(['tokens', 'timestamps', 'hours', 'doys', 'latent_shape', 'num_channels', 'split'])
{'tokens': tensor([[[[ 2.1545e-02,  4.7241e-02,  2.3718e-01,  ...,  2.5749e-05,
           -1.2494e-01, -1.3013e-01],
          [ 2.3682e-01, -3.1708e-02,  4.2084e-02,  ..., -2.6138e-02,
           -3.8892e-01, -1.9055e-01],
          [ 1.0693e-01,  1.6919e-01, -3.3252e-01,  ..., -1.5417e-01,
           -2.4445e-02, -2.0142e-01],
          ...,
          [ 1.2680e-02,  5.1483e-02,  1.2433e-01,  ..., -1.6736e-01,
           -2.6318e-01, -3.1738e-01],
          [ 2.2446e-02,  2.1960e-01,  1.4758e-01,  ...,  6.1188e-02,
           -4.2206e-02,  2.2412e-01],
          [ 2.7832e-02,  1.2225e-01,  3.1982e-01,  ..., -3.8177e-02,
           -2.1191e-01, -1.7236e-01]],

         [[-5.4245e-03, -4.1656e-03,  1.1673e-03,  ...,  8.1539e-04,
           -3.6106e-03, -9.6083e-04],
          [ 2.9325e-04,  1.0700e-03, -1.3340e-04,  ...,  9.0866e-03,
            1.3542e-03,  4.8904e-03],
          [ 4.1161e-03

In [12]:
print(data["tokens"].shape, "Data type: ",data["tokens"].dtype)
print(data["timestamps"])
print("Latent Shape: ",data["latent_shape"])
print("Number of Channels: ",data["num_channels"])

torch.Size([59900, 1024, 8, 15]) Data type:  torch.float16
['1979-01-01T00:00:00' '1979-01-01T06:00:00' '1979-01-01T12:00:00' ...
 '2019-12-31T06:00:00' '2019-12-31T12:00:00' '2019-12-31T18:00:00']
Latent Shape:  [1024, 8, 15]
Number of Channels:  1024


In [13]:
hours = data["hours"].numpy()
N = len(hours)
keep = np.where(hours == 0)[0] 

output = {}
for k, v in data.items():
    if hasattr(v, "shape") and len(v.shape) > 0 and v.shape[0] == N:   
        output[k] = v[torch.from_numpy(keep)] if torch.is_tensor(v) else v[keep]
    else:
        output[k] = v 

torch.save(output, "tokens_train_daily.pt")
print(len(keep), "of", N)  

14975 of 59900


In [16]:
daily = torch.load("./tokens_train_daily.pt", weights_only=False, map_location="cpu", mmap=True)
print(daily)
print(daily["timestamps"])

{'tokens': tensor([[[[ 2.1545e-02,  4.7241e-02,  2.3718e-01,  ...,  2.5749e-05,
           -1.2494e-01, -1.3013e-01],
          [ 2.3682e-01, -3.1708e-02,  4.2084e-02,  ..., -2.6138e-02,
           -3.8892e-01, -1.9055e-01],
          [ 1.0693e-01,  1.6919e-01, -3.3252e-01,  ..., -1.5417e-01,
           -2.4445e-02, -2.0142e-01],
          ...,
          [ 1.2680e-02,  5.1483e-02,  1.2433e-01,  ..., -1.6736e-01,
           -2.6318e-01, -3.1738e-01],
          [ 2.2446e-02,  2.1960e-01,  1.4758e-01,  ...,  6.1188e-02,
           -4.2206e-02,  2.2412e-01],
          [ 2.7832e-02,  1.2225e-01,  3.1982e-01,  ..., -3.8177e-02,
           -2.1191e-01, -1.7236e-01]],

         [[-5.4245e-03, -4.1656e-03,  1.1673e-03,  ...,  8.1539e-04,
           -3.6106e-03, -9.6083e-04],
          [ 2.9325e-04,  1.0700e-03, -1.3340e-04,  ...,  9.0866e-03,
            1.3542e-03,  4.8904e-03],
          [ 4.1161e-03,  7.6561e-03,  4.4899e-03,  ...,  6.7635e-03,
            9.0637e-03,  8.8043e-03],
         